In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("../UCI-dataset/diabetic_data.csv")

In [5]:
df = df.replace("?", np.nan)

In [6]:
df["readmitted_30"] = (
    df["readmitted"] == "<30"
).astype(int)

In [8]:
selected_features = [
    "age",
    "gender",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "insulin",
    "change",
    "diabetesMed"
]

In [9]:
X = df[selected_features]
y = df["readmitted_30"]

In [10]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["patient_nbr"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (81613, 15)
Testing shape: (20153, 15)


In [11]:
print("Train target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Train target distribution:
readmitted_30
0    88.719934
1    11.280066
Name: proportion, dtype: float64

Test target distribution:
readmitted_30
0    89.326651
1    10.673349
Name: proportion, dtype: float64


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [12]:
numerical_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

In [14]:
categorical_features = [
    "age",
    "gender",
    "admission_type_id",
    "admission_source_id",
    "insulin",
    "change",
    "diabetesMed"
]

In [18]:
numerical_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)